Come sappiamo in IR l'obiettivo ultimo è riuscire a restituire **documenti rilevanti**, ossia che soddisfino il bisogno informativo dell'utente.

Abbiamo anche visto che per valutare le performance di un sistema di IR è necessario avere a disposizione un **benchmark** composto da tre elementi fondamentali:
1. **Collezione di documenti**
2. **Un insieme di query di test**
3. **Un giudizio di rilevanza per ogni coppia query-documento**

Abbiamo studiato diverse misure di valutazione, le tre base sono:
- **Precision**: misura quanti documenti tra quelli recuperati sono rilevanti:
    $$Precision = \frac{\text{documenti rilevanti}}{\text{documenti recuperati}} = \frac{TP}{TP + FP}$$
- **Recall**: misura quanti documenti rilevanti tra tutti i documenti della collezione sono stati recuperati:
    $$Recall = \frac{\text{documenti rilevanti}}{\text{documenti rilevanti totali}} = \frac{TP}{TP + FN}$$
- **F1-score**: è la media armonica tra precision e recall. Rappresenta una specie di soft minimum, in quanto la media armonica tende ad essere bassa anche se uno solo dei due valori è basso:
    $$F1 = \frac{2PR}{P + R}$$

**Problema**: spesso precision e recall sono in conflitto tra loro, se voglio aumentare molto la recall recupero più documenti, ma rischio così di includere molti documenti non rilevanti abbassando la precision e viceversa.

In questa lezione ci concentreremo in particolare su **come migliorare la Recall**. In questo senso il problema che si affronta è quello per cui un utente potrebbe scrivere una query troppo semplice, usando parole diverse da quelle presenti nei documenti rilevanti e quindi non riuscire a recuperarli con un sistema di retrieval semplice.

es. query: "aircraft" --> un documento rilevante potrebbe contenere "plane", ma non "aircraft". Dal momento che i sistemi di IR visti finora si basano strettamente sul match dei termini, allora questo non riuscirebbe a recuperare quel documento nonostante sia effettivamente rilevante per il bisogno informativo dell'utente.

**Come recuperare documenti rilevanti anche quando non contengono esattamente le parole della query originale?** La strategia che vedremo per risolvere il problema è la **query expansion**: l'idea è **espandere la query iniziale aggiungendo nuovi termini utili**

es. di prima: se l'utente cerca aircraft, il sistema potrebbe aggiungere automaticamente alla query plane, airplane, aviation, flight etc..

Esistono due modi principali per fare questa espansione:
1. **Metodo Locale == Relevance Feedback**: il sistema guarda queli documenti l'utente ha giudicato rilevanti e li sfrutta per migliorare la query
2. **Metodo Globale == Global Query Expansion**: il sistema sfrutta l'intera collezione di documenti in una volta sola, per costruire un **thesaurus** a partire dal quale espandere le query.

**Thesaurus** == una rete di termini che sono in qualche modo correlati tra loro, ad esempio perché sono sinonimi, iperonimi (termine che racchiude in sé termini più specifici, detti iponimi; es. di iperonimo è "animale" che racchiude in sé iponimi come "cane", "gatto", "uccello" etc.) etc..


### Relevance Feedback
Il relevance feedback funziona come segue:
1. L'utente fa una query iniziale, spesso breve
2. Il motore di ricerca restituisce una lista di documenti ordinati per rilevanza
3. L'utente marca alcuni documenti come rilevanti e altri come non rilevanti
4. Il sistema usa queste informazioni per espandere la query iniziale, ad esempio aggiungendo termini **importanti** (quelli con alto peso tf-idf per quel documento) che compaiono nei documenti rilevanti
5. Il sistema rilancia la query modificata
6. I nuovi risultati dovrebbero avere recall migliore

Il processo potrebbe anche **essere iterato più volte**: l'utente dà feedback, il sistema espande la query, l'utente dà nuovo feedback etc. fino a quando non si raggiunge un risultato soddisfacente.

es. query iniziale "new space satellite applications". Il sistema restituisce una lista di risultati, da cui l'utente marca alcuni documenti come rilevanti (simbolo +). Dopodiché il sistema guarda il contenuto dei documenti in questione e vede che compaiono termini importanti come "nasa", "launch", "earth" etc.. quindi li utilizza per costruire una query espansa che permetterò di far salire altri documenti rilevanti

<p align="center">
    <img src="img/c11.png" alt="Relevance Feedback" width=45%>
    <img src="img/c12.png" alt="Relevance Feedback" width=40%>
</p>

Prima di passare a capire la teoria dietro il relevance feedback e un algoritmo per implementarlo, è importante sottolineare che questa tecnica come idea generale può funzionare con tutti i modelli che abbiamo visto finora. 

L'algoritmo che vedremo però (Rocchio) è pensato, nella sua forma classica, solo per VSM.

#### Analisi teorica del relevance feedback
Per capire formalmente il relevance feedback, dobbiamo introdurre il concetto di **centroide**. Nel modello vettoriale, sappiamo che ogni documento **è rappresentato come un vettore in uno spazio ad alta dimensionalità, e che ogni dimensione corrisponde a un termine**. 

Quindi ogni documento e la query sono vettori nello spazio, e possiamo confrontarli tra loro usando misure di similarità come la cosine similarity. 

Il **centroide** di un certo insieme di documenti rappresenta **il punto medio di quei documenti nello spazio vettoriale**. Formalmente, sia $D$ un insieme di documenti, allora il centroide di $D$ è dato da:
$$ \mu(D) = \frac{1}{|D|} \sum_{d \in D} \vec{d} $$
Immaginiamo di conoscere tutti i documenti rilevanti ($C_r$) e non rilevanti ($C_{nr}$) per una certa query. Allora la query ideale dovrebbe essere posizionata in modo da essere vicina ai documenti rilevanti e lontana da quelli non rilevanti, quindi intuitivamente dovrebbe essere posizionata in una zona dello spazio che è vicina al centroide dei documenti rilevanti e lontana da quello dei documenti non rilevanti:
$$\vec{q}_\text{opt} = \mu(C_r) - \mu(C_{nr}) $$
Naturalmente però se conoscessimo tutti i documenti rilevanti e non rilevanti sarebbe un cazzo e tutt'uno, avremmo già risolto il problema. Per questo introduciamo l'algoritmo di Rocchio.

#### Algoritmo di Rocchio e Assunzioni di RF
L'algoritmo di Rocchio è il metodo classico per fare relevance feedback in VSM. Si parte dalla query originale $q_0$ e la espande usando **sia i documenti giudicati rilevanti che quelli giudicati non rilevanti**. La formula è la seguente:
$$\vec{q}_m = \alpha \vec{q}_0 + \beta \mu(D_r) - \gamma \mu(D_{nr}) = \alpha \vec{q}_0 + \beta \frac{1}{|D_r|} \sum_{d \in D_r} \vec{d} - \gamma \frac{1}{|D_{nr}|} \sum_{d \in D_{nr}} \vec{d}$$
dove:
- $\vec{q}_m$ è la query modificata
- $\vec{q}_0$ è la query originale
- $D_r$ è l'insieme dei documenti marcati rilevanti
- $D_{nr}$ è l'insieme dei documenti marcati non rilevanti
- $\alpha$, $\beta$, $\gamma$ sono parametri che controllano l'importanza relativa della query originale, dei documenti rilevanti e di quelli non rilevanti

**Quando funziona relevance feedback?** Si enunciano due assunzioni importanti:
1. **Assunzione A1**: *l'utente deve conoscere abbastanza bene il vocabolario della collezione per scrivere una query iniziale decente*.  
Se l'utente infatti usasse parole totalmente diverse da quelle presenti nei documenti, allora già all'inizio il sistema non riuscirebbe a recuperare documenti utili e quindi non avrebbe materiale su cui basarsi per espandere la query.  
es. cosmonaut/astronaut --> se l'utente cerca cosmonaut, ma la collezione usa principalmente astronaut --> problematico.
1. **Assunzione A2**: *i documenti rilevanti devono essere abbastanza simili tra loro*. Il relevance feedback funziona bene se i documenti rilevanti condividono termini o argomenti simili, ma può dare problemi se una stessa query ha più "prototipi" diversi di documenti rilevanti.  
es. query = "personaggio con doppia vita". Questa query può avere documenti rilevanti molto diversi tra loro; alcuni potrebbero parlare di Dr. Jekyll e Mr. Hyde, altri di Batman, altri ancora di una spia o un agente segreto. Il problema è che tutti questi documenti non usano necessariamente gli stessi termini --> se l'utente marca come rilevante quello di Batman, il sistema gli suggerirà roba legata a Gotham, Joker, pipistrelli etc.. peggiorando rispetto agli altri argomenti.

#### Problemi del relevance feedback e Pseudo-Relevance Feedback
Il relevance feedback ha molti problemi pratici: 
1. **è costoso**: le query modificate diventano più lunghe, quindi più costose da processare
2. **Gli utenti non sono sempre disposti a dare feedback**: molte persone spesso non hanno voglia di segnare manualmente i risultati rilevanti e non rilevanti rispetto alla loro query
3. **Può essere difficile capire perché un certo documento viene recuperato dopo il feedback**: infatti la query modificata diventa spesso lunga e poco interpretabile

Riguardo il secondo problema, per evitare di chiedere il feedback manuale all'utente, si può usare il **Pseudo-Relevance Feedback**. L'idea è la seguente:
1. l'utente scrive una query 
2. il sistema recupera i primi risultati
3. il sistema **assume automaticamente che i primi $k$ documenti siano rilevanti**, usando quei documenti come se fossero feedback positivo
4. il sistema espande la query e rilancia la ricerca

Il rischio dello Pseudo-Relevance Feedback è che se i primi $k$ documenti recuperati non sono effettivamente rilevanti, allora si rischia di espandere la query in modo errato, peggiorando le performance invece di migliorarle. Si parla in questo senso di **Query Drift**.

Il **Query Drift** avviene quando la query modificata si allontana dal vero bisogno informativo dell'utente. Ad esempio, se l'utente cerca "satellite applications" ma i primi risultati parlano molto di NASA o di clima, allora il sistema può espandere la query con termini che la spostano verso un altro tema. **Ciò può portarela query, dopo una o più iterazioni, a derivare sempre più lontano dal bisogno informativo originale**.

Per questi motivi lo pseudo-relevance feedback può funzionare molto bene in media, ma può anche fare malissimo per alcune query.